In [1]:
!pip install networkx


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import json
import os
from collections import defaultdict, deque
from datetime import datetime, timedelta, timezone
import networkx as nx

DATASET_DIR  = "dataset"
ALERTS_FILE  = os.path.join(DATASET_DIR, "alerts_sample.jsonl")
SERVICES_FILE = os.path.join(DATASET_DIR, "services.json")
RESULTS_DIR  = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)



---
## Cell 2 — Load Dataset

Load `alerts_sample.jsonl` (20 alert) và `services.json` (service graph).

In [ ]:
alerts = []
with open(ALERTS_FILE, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            alerts.append(json.loads(line))
print(f"Loaded {len(alerts)} alerts")

Loaded 20 alerts
a-0001 2026-06-12T09:42:01Z payment-svc db_connection_pool_used_ratio warn 0.85
a-0002 2026-06-12T09:42:18Z payment-svc db_connection_pool_used_ratio crit 0.99
a-0003 2026-06-12T09:42:22Z payment-svc latency_p99_ms crit 1840
a-0004 2026-06-12T09:42:30Z payment-svc error_rate warn 0.04
a-0005 2026-06-12T09:42:45Z checkout-svc latency_p99_ms warn 2100
a-0006 2026-06-12T09:43:01Z checkout-svc downstream_payment_error_rate crit 0.06
a-0007 2026-06-12T09:43:15Z edge-lb upstream_5xx_rate warn 0.025
a-0008 2026-06-12T09:43:18Z payment-svc latency_p99_ms crit 1840
a-0009 2026-06-12T09:43:32Z cart-svc latency_p99_ms warn 920
a-0010 2026-06-12T09:43:50Z notification-svc queue_lag_ms warn 4200
a-0011 2026-06-12T09:44:02Z payment-svc db_connection_pool_used_ratio crit 1.0
a-0012 2026-06-12T09:44:30Z checkout-svc request_drop_rate crit 0.12
a-0013 2026-06-12T09:45:10Z recommender-svc cpu_utilization warn 0.91
a-0014 2026-06-12T09:45:42Z edge-lb p99_latency_ms crit 3500
a-0015 2026-

In [ ]:
#Layer 1: fingerprint
def fingerprint(alert: dict) -> str:
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

class Deduper:
    def __init__(self):
        self.store: dict[str, dict] = {}  # fingerprint → cluster info

    def push(self, alert: dict) -> str:
        fp = fingerprint(alert)
        ts = datetime.fromisoformat(alert['ts'].replace('Z', '+00:00'))

        if fp not in self.store:
            self.store[fp] = {
                'cluster_id': fp,
                'count': 1,
                'first_seen': ts,
                'last_seen': ts,
                'alerts': [alert['id']],
                'max_value': alert['value'],
            }
        else:
            c = self.store[fp]
            c['count'] += 1
            c['last_seen'] = ts
            c['alerts'].append(alert['id'])
            c['max_value'] = max(c['max_value'], alert['value'])

        return fp

    def clusters(self) -> list[dict]:
        return list(self.store.values())


deduper = Deduper()
for a in alerts:
    deduper.push(a)

dedup_result = deduper.clusters()
print(f"  Input : {len(alerts)} alerts")
print(f"  Output: {len(dedup_result)} unique fingerprints")
for c in sorted(dedup_result, key=lambda x: -x['count']):
    dup = "← DUPLICATE" if c['count'] > 1 else ""
    print(f"  {c['count']:3d}   {c['cluster_id']:55s} {dup}")

Layer 1 — Dedup:
  Input : 20 alerts
  Output: 17 unique fingerprints
    3   payment-svc|latency_p99_ms|crit                         ← DUPLICATE
    2   payment-svc|db_connection_pool_used_ratio|crit          ← DUPLICATE
    1   payment-svc|db_connection_pool_used_ratio|warn          
    1   payment-svc|error_rate|warn                             
    1   checkout-svc|latency_p99_ms|warn                        
    1   checkout-svc|downstream_payment_error_rate|crit         
    1   edge-lb|upstream_5xx_rate|warn                          
    1   cart-svc|latency_p99_ms|warn                            
    1   notification-svc|queue_lag_ms|warn                      
    1   checkout-svc|request_drop_rate|crit                     
    1   recommender-svc|cpu_utilization|warn                    
    1   edge-lb|p99_latency_ms|crit                             
    1   search-svc|catalog_db_query_time_ms|warn                
    1   checkout-svc|latency_p99_ms|crit                       

In [20]:
def session_groups(alerts, gap_sec) -> list[list[dict]]:
    if not alerts:
        return []
    sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
    groups = [[sorted_alerts[0]]]

    for alert in sorted_alerts[1:]:
        ts = datetime.fromisoformat(alert['ts'].replace('Z', '+00:00'))
        last_ts = datetime.fromisoformat(groups[-1][-1]['ts'].replace('Z', '+00:00'))

        if (ts - last_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)   
        else:
            groups.append([alert])

    return groups


In [22]:
def build_graph(services_json_path: str) -> nx.DiGraph:
    """
    Build directed graph: A → B nghĩa là A gọi B.

    Khi RCA traverse, bạn sẽ đi NGƯỢC edge (từ A về B) — vì nếu A alert
    thì có thể B là root cause.
    """
    g = nx.DiGraph()
    data = json.loads(open(services_json_path, encoding="utf-8").read())  # ← fix ở đây

    # Add service nodes
    for svc in data['services']:
        g.add_node(svc['name'], **{k: v for k, v in svc.items() if k != 'name'})

    # Add store nodes (DB, Redis, Kafka)
    for store in data['stores']:
        g.add_node(store['name'], **{k: v for k, v in store.items() if k != 'name'})

    # Add edges
    for edge in data['edges']:
        g.add_edge(edge['from'], edge['to'], type=edge['type'])

    return g


# ── Build graph ───────────────────────────────────────────────────────────
graph = build_graph(SERVICES_FILE)

print(f"Service Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print()
print("Edges (A → B: A gọi B):")
for u, v, d in graph.edges(data=True):
    print(f"  {u:20s} →  {v:20s}  [{d['type']}]")

# ── Minh hoạ cascade path (Section 5.1) ──────────────────────────────────
print()
print("Cascade path (undirected — Section 5.3):")
undirected = graph.to_undirected()
for pair in [("payment-svc", "checkout-svc"),
             ("payment-svc", "edge-lb"),
             ("payment-svc", "cart-svc"),
             ("payment-svc", "recommender-svc"),
             ("payment-svc", "search-svc"),
             ("payment-svc", "notification-svc")]:
    s1, s2 = pair
    try:
        d = nx.shortest_path_length(undirected, s1, s2)
        p = nx.shortest_path(undirected, s1, s2)
        flag = "✓ gộp" if d <= 2 else "✗ tách"
        print(f"  {s1} ↔ {s2}: {d} hop {flag}")
        print(f"    path: {' → '.join(p)}")
    except nx.NetworkXNoPath:
        print(f"  {s1} ↔ {s2}: không có path ✗ tách")

Service Graph: 14 nodes, 17 edges

Edges (A → B: A gọi B):
  edge-lb              →  auth-svc              [http]
  edge-lb              →  catalog-svc           [http]
  edge-lb              →  search-svc            [http]
  edge-lb              →  checkout-svc          [http]
  checkout-svc         →  cart-svc              [http]
  checkout-svc         →  payment-svc           [http]
  checkout-svc         →  inventory-svc         [http]
  checkout-svc         →  notification-svc      [kafka]
  payment-svc          →  payments-db           [postgres]
  cart-svc             →  cart-redis            [redis]
  cart-svc             →  catalog-svc           [http]
  catalog-svc          →  catalog-db            [postgres]
  catalog-svc          →  recommender-svc       [http]
  recommender-svc      →  catalog-db            [postgres]
  inventory-svc        →  catalog-db            [postgres]
  notification-svc     →  kafka-events          [kafka]
  search-svc           →  catalog-db      

In [23]:
def topology_group(alerts, graph, max_hop = 2) -> list[list[dict]]:
    """
    Group alerts nếu service của chúng cách nhau <= max_hop trên service graph.
    """
    if not alerts:
        return []

    undirected = graph.to_undirected()
    by_service = defaultdict(list)
    for a in alerts:
        by_service[a['service']].append(a)

    services_with_alerts = list(by_service.keys())
    parent = {s: s for s in services_with_alerts}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]  # path compression
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    # Two services cùng group nếu khoảng cách <= max_hop trên graph
    for i, s1 in enumerate(services_with_alerts):
        for s2 in services_with_alerts[i+1:]:
            try:
                dist = nx.shortest_path_length(undirected, s1, s2)
                if dist <= max_hop:
                    union(s1, s2)
            except nx.NetworkXNoPath:
                continue  # 2 service không connected → không group

    # Collect groups
    groups_dict = defaultdict(list)
    for s in services_with_alerts:
        groups_dict[find(s)].extend(by_service[s])

    return list(groups_dict.values())

In [27]:
GAP_SEC = 120   
MAX_HOP = 2     
def correlate(alerts, graph, gap_sec, max_hop):
    sessions = session_groups(alerts, gap_sec=gap_sec)

    all_clusters = []
    for session_idx, session_alerts in enumerate(sessions):
        # Trong mỗi session, topology-group
        topo_groups = topology_group(session_alerts, graph, max_hop=max_hop)
        for group_idx, group in enumerate(topo_groups):
            all_clusters.append({
                'cluster_id':   f'c-{session_idx:03d}-{group_idx:03d}',
                'alert_count':  len(group),
                'services':     sorted(set(a['service'] for a in group)),
                'alert_ids':    [a['id'] for a in group],
                'time_range':   [min(a['ts'] for a in group), max(a['ts'] for a in group)],
                'max_severity': max(a['severity'] for a in group),
                # fingerprints: track dedup info (unique alert types trong cluster)
                'fingerprints': sorted(set(fingerprint(a) for a in group)),
            })

    return all_clusters

clusters = correlate(alerts, graph, gap_sec=GAP_SEC, max_hop=MAX_HOP)
reduction = round(1 - len(clusters) / len(alerts), 4)
print(f"Input  : {len(alerts)} alerts")
print(f"Output : {len(clusters)} clusters")
print(f"Reduction ratio: {reduction} ({reduction*100:.1f}%)")
print("ĐẠT" if reduction >= 0.5 else "CHƯA ĐẠT")
print()
for c in clusters:
    sev = c['max_severity']
    print(f"[{c['cluster_id']}] {c['alert_count']} alerts | severity={sev}")
    print(f"  services  : {c['services']}")
    print(f"  time      : {c['time_range'][0]} → {c['time_range'][1]}")
    print(f"  alert_ids : {c['alert_ids']}")
    print(f"  fingerprints:")
    for fp in c['fingerprints']:
        print(f"    - {fp}")
    print()

Input  : 20 alerts
Output : 1 clusters
Reduction ratio: 0.95 (95.0%)
ĐẠT

[c-000-000] 20 alerts | severity=warn
  services  : ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']
  time      : 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
  alert_ids : ['a-0001', 'a-0002', 'a-0003', 'a-0004', 'a-0008', 'a-0011', 'a-0015', 'a-0018', 'a-0005', 'a-0006', 'a-0012', 'a-0017', 'a-0007', 'a-0014', 'a-0020', 'a-0009', 'a-0010', 'a-0019', 'a-0013', 'a-0016']
  fingerprints:
    - cart-svc|latency_p99_ms|warn
    - checkout-svc|downstream_payment_error_rate|crit
    - checkout-svc|latency_p99_ms|crit
    - checkout-svc|latency_p99_ms|warn
    - checkout-svc|request_drop_rate|crit
    - edge-lb|p99_latency_ms|crit
    - edge-lb|upstream_5xx_rate|crit
    - edge-lb|upstream_5xx_rate|warn
    - notification-svc|queue_depth|crit
    - notification-svc|queue_lag_ms|warn
    - payment-svc|db_connection_pool_used_ratio|crit
    - payment-svc|db_conne